In [4]:
from qdrant_client.http.models import Distance, VectorParams, SparseVectorParams
from app.config import COLLECTION_NAME, CHUNKS_FILE, device
from app.utils.chunking import load_chunks
from app.ingest.qdrant_factory import QdrantFactory

In [5]:
def ingest_chunks_to_qdrant():
    """
    Ingests pre-chunked documents into Qdrant vector store using local embeddings.
    Creates the collection if it doesn't exist and adds texts + metadata in batch.
    """
    # 1. Initialize factory with device
    factory = QdrantFactory(device=device)

    # 2. Use factory to get client and vector store
    client = factory.client

    # 3. Create collection if it doesn't exist (fixed dimension + named vectors)
    if not client.collection_exists(collection_name=COLLECTION_NAME):
        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config={
                "text-dense": VectorParams(size=384, distance=Distance.COSINE)
            },
            sparse_vectors_config={
                "text-sparse": SparseVectorParams()  # no size needed for sparse vectors
            },
        )
        print(f"Collection '{COLLECTION_NAME}' created with 'text-dense' vector.")
    else:
        print(f"Collection '{COLLECTION_NAME}' already exists.")

    # Check if collection exists before ingesting
    vector_store = factory.get_qdrant_vector_store()

    # 4. Load chunks from pickle file (saved by extract_3gpp.ipynb)
    chunks = load_chunks(CHUNKS_FILE)
    if not chunks:
        print("No chunks found to ingest.")
        return

    # 5. Prepare texts and metadatas
    texts = []
    metadatas = []

    for index, chunk in enumerate(chunks):
        text = chunk.get("content", "")
        if not text:
            continue

        texts.append(text)

        metadata = {
            "release": chunk.get("release", ""),
            "series": chunk.get("series", ""),
            "spec": chunk.get("spec", ""),
            "chunk_index": index,  # opcional
        }
        metadatas.append(metadata)

    if not texts:
        print("No valid texts found after processing.")
        return

    # 6. Batch ingest
    vector_store.add_texts(texts=texts, metadatas=metadatas)
    print(f"Ingested {len(texts)} chunks into collection '{COLLECTION_NAME}'.")

In [6]:
# Run ingestion
ingest_chunks_to_qdrant()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1299.69it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection '3gpp_rel18_28' already exists.
Chunks loaded from ../../../files/chunks/tspec_chunks_test_rel_18_28.pkl
Ingested 9102 chunks into collection '3gpp_rel18_28'.


# Test collection

In [7]:
factory = QdrantFactory(device=device)
client = factory.client
collection_info = client.get_collection(COLLECTION_NAME)
print(collection_info)
print(f"\nCollection: {COLLECTION_NAME}")
print(f"Status: {collection_info.status}")
print(f"Points count: {collection_info.points_count:,}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1704.75it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=14982 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors={'text-dense': VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors={'text-sparse': SparseVectorParams(index=None, modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=Non